# Task 5: Character-Level Text Generation Engine
### 🤖 Deep Learning & Sequence Modeling — CodSoft Internship Submission

---

## 🎯 Project Objective
The objective of this project is to implement a **Character-Level Recurrent Neural Network (RNN)** capable of capturing language mechanics, syntax, and specialized engineering vocabulary. 

Instead of treating text generation at the word level, this neural network processes sequences **character-by-character**. It calculates structural dependencies from the input text corpus to learn underlying distributions, allowing it to predict and generate entirely new, coherent text strings based on what it has learned.

### 📊 Dataset Origin & Strategic Context
To elevate this project from a standard tutorial pipeline, the model is trained directly on a 35-page financial machine learning research publication: **"Trading Agents: Multi-Agents LLM Financial Trading Framework"** (Source: Hugging Face/arXiv). 
* **Corpus Density:** Over 50,000+ characters of highly dense academic and engineering text.
* **Lexical Challenge:** The network must learn to parse and replicate complex terminology like `multi-agent`, `LLM`, `technical indicators`, `Sharpe ratio`, and statistical modeling layouts.

In [17]:
import pypdf
import torch
import torch.nn as nn
import numpy as np
import os

In [18]:
# =====================================================================
# 1. READ AND CONVERT HUGGING FACE PDF TO TEXT CORPUS
# =====================================================================
pdf_path = "Handwritten text.pdf"
raw_text_corpus = ""

if os.path.exists(pdf_path):
    print(f"Reading from dataset: '{pdf_path}'...")
    reader = pypdf.PdfReader(pdf_path)
    for i, page in enumerate(reader.pages):
        text_chunk = page.extract_text()
        if text_chunk:
            raw_text_corpus += text_chunk + "\n"
    print(f"Successfully extracted {len(raw_text_corpus)} characters from the document.")
else:
    raise FileNotFoundError(f"Could not find '{pdf_path}' in the current working directory. Please place the file in the same folder.")

# Standardize whitespace transitions
text_corpus = " ".join(raw_text_corpus.split())

# Create character level sequence mapping arrays
chars = sorted(list(set(text_corpus)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Unique structural tokens (Vocabulary Size): {vocab_size}")

Reading from dataset: 'Handwritten text.pdf'...
Successfully extracted 113250 characters from the document.
Unique structural tokens (Vocabulary Size): 101


## ⚙️ Hyperparameters & Training Constraints

To achieve stable convergence across the academic text corpus, the training pipeline was configured using the following parameters:

* **Sequence Length (`seq_length = 50`)**
  
  Defines the look-back window used during training. The model observes 50 consecutive characters and learns to predict the 51st character in the sequence.

* **Loss Function (`nn.CrossEntropyLoss`)**
  
  Treats next-character prediction as a multi-class classification problem across the entire character vocabulary, penalizing incorrect probability assignments during training.

* **Optimization Engine (`Adam Optimizer`)**
  
  Uses an adaptive learning rate strategy with:

  $$
  lr = 0.003
  $$

  This enables efficient gradient updates and promotes stable convergence throughout training.

* **Target Vector Shifting**
  
  For each training sequence, the target vector is shifted exactly **one character position to the right** of the input vector. This teaches the network to continuously predict the next immediate character in a sequence.

### 📋 Final Training Configuration

| Hyperparameter | Value |
|---------------|--------|
| Sequence Length | 50 |
| Embedding Dimension | 64 |
| Hidden Units | 128 |
| RNN Layers | 2 |
| Learning Rate | 0.003 |
| Optimizer | Adam |
| Loss Function | CrossEntropyLoss |
| Training Epochs | 1500 |

### 🎯 Training Objective

Given an input sequence:

```text
Input  : "Trading agent framework is"
Target : "rading agent framework is "
```

the model learns to predict each subsequent character, gradually capturing spelling patterns, grammar structures, and domain-specific terminology from the training corpus.

In [19]:
# =====================================================================
# 2. DEVICE INITIALIZATION
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing Deep Learning Engine Context: {device}")

Executing Deep Learning Engine Context: cpu


In [20]:
# =====================================================================
# 3. CONSTRUCT CHARACTER-LEVEL RNN ARCHITECTURE
# =====================================================================
class CharacterGenerativeRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2):
        super(CharacterGenerativeRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Continuous embedding dimension mapping
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Recurrent Network Cell Architecture
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)
        # Fully connected transformation out to token space
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden):
        out = self.embedding(x)
        out, hidden = self.rnn(out, hidden)
        out = out.reshape(-1, self.hidden_dim)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)

# Model structural parameters
embedding_dim = 64
hidden_dim = 128
model = CharacterGenerativeRNN(vocab_size, embedding_dim, hidden_dim, num_layers=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [21]:
# =====================================================================
# 4. ITERATIVE SEQUENCE TRAINING PIPELINE
# =====================================================================
print("\nInitiating sequential training matrix over the paper's corpus...")
seq_length = 50
epochs = 1500

for epoch in range(1, epochs + 1):
    # Slice a random baseline window from the PDF text contents
    start_point = np.random.randint(0, len(text_corpus) - seq_length - 1)
    chunk = text_corpus[start_point : start_point + seq_length + 1]
    
    # Input tensor vs shifted Target sequence characters
    input_seq = torch.tensor([char_to_idx[ch] for ch in chunk[:-1]], dtype=torch.long).unsqueeze(0).to(device)
    target_seq = torch.tensor([char_to_idx[ch] for ch in chunk[1:]], dtype=torch.long).to(device)
    
    hidden = model.init_hidden(batch_size=1)
    model.zero_grad()
    
    output, hidden = model(input_seq, hidden)
    loss = criterion(output, target_seq)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 250 == 0:
        print(f"Epoch [{epoch}/{epochs}] ---- Evaluated System Loss Metric: {loss.item():.4f}")


Initiating sequential training matrix over the paper's corpus...
Epoch [250/1500] ---- Evaluated System Loss Metric: 2.7507
Epoch [500/1500] ---- Evaluated System Loss Metric: 2.2818
Epoch [750/1500] ---- Evaluated System Loss Metric: 2.2612
Epoch [1000/1500] ---- Evaluated System Loss Metric: 1.4812
Epoch [1250/1500] ---- Evaluated System Loss Metric: 2.3247
Epoch [1500/1500] ---- Evaluated System Loss Metric: 1.8349


## 🔮 Softmax Temperature-Scaled Text Generation Inference

Once training wraps up, the model is switched to evaluation mode (`model.eval()`). To avoid repetitive, predictable loops (where the model endlessly picks only the absolute single highest probability character), we deploy a **Temperature-Scaled Multinomial Sampling** technique.

$$P(c_i) = \frac{e^{\frac{z_i}{T}}}{\sum_{j} e^{\frac{z_j}{T}}}$$

* **Temperature parameter ($T = 0.75$):** Controls the structural creativity of the text output.
  * A **lower temperature (< 0.5)** forces strict, safe, and highly repetitive predictions.
  * A **higher temperature (> 1.0)** introduces random structural noise, increasing creativity but risking spelling errors.
  * Setting it to **0.75** balances logical syntax with structural diversity.

In [22]:
# =====================================================================
# 5. GENERATIVE TEXT INFERENCE ENGINE
# =====================================================================
def generate_inference_text(model, seed_str="Trading ", gen_len=120, temperature=0.75):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([char_to_idx[ch] for ch in seed_str], dtype=torch.long).unsqueeze(0).to(device)
        hidden = model.init_hidden(1)
        
        generated_result = seed_str
        current_input = input_tensor
        
        for _ in range(gen_len):
            output, hidden = model(current_input[:, -1:], hidden)
            
            # Use temperature scaling to avoid predictive repetitive loops
            distribution = output.data.view(-1).div(temperature).exp()
            predicted_idx = torch.multinomial(distribution, 1)[0].item()
            
            generated_result += idx_to_char[predicted_idx]
            current_input = torch.tensor([[predicted_idx]], dtype=torch.long).to(device)
            
        return generated_result

print("\n=== GENERATING TEXT PATTERNS INSPIRED BY TRAINING CORPUS ===")
print(generate_inference_text(model, seed_str="Trading ", gen_len=150))


=== GENERATING TEXT PATTERNS INSPIRED BY TRAINING CORPUS ===
Trading ster teral dumbing inalerualyst einpire to devermen ta day S e n i s in and repte LI in ot en ti en i n c a l e s t _ f e r e d t i e d u n g e n t i 
